In [ ]:
import numpy as np
from scipy.linalg import inv
from pandas import read_excel

# ==========================================
# 定义通式函数 ( The General Solver )
# ==========================================
def solve_vibration(M, C, K, f_hz, force_pp, dof_force, dof_resp):
    """
    Inputs:
        M, C, K   : 系统矩阵 (System Matrices)
        f_hz      : 频率 (Hz)
        force_pp  : 力的峰峰值 (N, Peak-to-Peak)
        dof_force : 施力位置 (DOF 1-8, Human Index)
        dof_resp  : 测量位置 (DOF 1-8, Human Index)
    Returns:
        disp_pp   : 位移峰峰值 (m)
        phase_deg : 相位角 (Degrees)
    """
    
    # 1. 自动修正索引 (Human Index -> Python Index)
    # 输入 DOF 3 (1-based) -> 变成 index 2 (0-based)
    idx_in = dof_force - 1
    idx_out = dof_resp - 1
    
    omega = 2 * np.pi * f_hz
    F_amp = force_pp / 2  # 峰峰值 -> 幅值
    
    # 3. 核心计算 (Core Calculation: X = H * F)
    # Z = Dynamic Stiffness (动态刚度)
    Z = K - (omega**2 * M) + (1j * omega * C)
    
    # H = FRF Matrix (这里求逆)
    H = inv(Z)
    
    # 提取特定的传递函数值 H[out, in]
    h_value = H[idx_out, idx_in]
    
    # 计算复数位移 (Complex Displacement)
    X_complex = h_value * F_amp
    
    # 4. 后处理 (Post-Processing)
    # 幅值变回峰峰值
    disp_pp = 2 * np.abs(X_complex)
    
    # 计算相位 (弧度变角度)
    phase_rad = np.angle(X_complex)
    phase_deg = np.degrees(phase_rad)
    
    return disp_pp, phase_deg

# ==========================================
# 使用示例 ( Usage )
# ==========================================

# 确保你的 M, C, K 已经加载好了！

K = read_excel('stiffness_matrix.xlsx')
M = read_excel('mass_matrix.xlsx')
C = read_excel('damping_matrix.xlsx')

# --- Case 1: 8N at 1.0 Hz, Force at 7, Measure at 3 ---
d1_m, phase1 = solve_vibration(M, C, K, f_hz=1.0, force_pp=8.0, dof_force=7, dof_resp=3)

print(f"--- Case 1 ---")
print(f"Displacement: {d1_m:.6e} m (p-p)")
print(f"Displacement: {d1_m * 1000:.4f} mm (p-p)") # 自动转 mm


# --- Case 2: 16N at 2.85 Hz, Force at 3, Measure at 3 ---
d2_m, phase2 = solve_vibration(M, C, K, f_hz=2.85, force_pp=16.0, dof_force=3, dof_resp=3)

print(f"\n--- Case 2 ---")
print(f"Displacement: {d2_m:.6e} m (p-p)")
print(f"Displacement: {d2_m * 1000:.4f} mm (p-p)")
print(f"Phase Angle : {phase2:.2f} degrees")

--- Case 1 ---
Displacement: 6.103259e-03 m (p-p)
Displacement: 6.1033 mm (p-p)

--- Case 2 ---
Displacement: 1.671001e-02 m (p-p)
Displacement: 16.7100 mm (p-p)
Phase Angle : -34.12 degrees


In [3]:
# ==========================================
# 多点受力通用解法 (Multi-Force Solver)
# ==========================================

def solve_multi_force(M, C, K, f_hz, forces_dict, dof_resp):
    """
    forces_dict: 一个字典，告诉我是哪些点受力，受多大的力。
                 格式：{DOF_ID: Force_PP_Value}
                 例如：{3: 10.0, 7: 20.0} 表示 DOF 3 受 10N，DOF 7 受 20N。
    """
    
    # 1. 准备频率和矩阵
    omega = 2 * np.pi * f_hz
    Z = K - (omega**2 * M) + (1j * omega * C)
    H = inv(Z) # 获取完整的 FRF 矩阵 (8x8)
    
    # 2. 构建力向量 (Force Vector)
    # 系统的自由度数量
    N = M.shape[0] 
    F_vec_amp = np.zeros(N) # 先创建一个全是 0 的向量 [0, 0, ... 0]
    
    # 把字典里的力填进去
    for dof, force_pp in forces_dict.items():
        # 记住：人类数数(DOF) -> Python索引(Index) 要减 1
        idx = dof - 1
        # 记住：峰峰值 -> 幅值 要除以 2
        F_vec_amp[idx] = force_pp / 2
        
    # 3. 矩阵乘法 (Matrix Multiplication)
    # 这里的 @ 符号就是矩阵乘法的意思： X = H * F
    X_vec_complex = H @ F_vec_amp 
    
    # 4. 提取结果
    # 找到我们要观察的那个测点 (dof_resp)
    idx_resp = dof_resp - 1
    X_target_complex = X_vec_complex[idx_resp]
    
    # 5. 后处理 (Amplitude -> Peak-to-Peak)
    disp_pp = 2 * np.abs(X_target_complex)
    
    return disp_pp

# ================= Use Example =================

# 假设题目：2.5 Hz, DOF 3 受 10N, DOF 7 受 20N, 测 DOF 3 的位移
my_forces = {
    3: 10.0,
    7: 20.0
}

result_pp = solve_multi_force(M, C, K, f_hz=2.5, forces_dict=my_forces, dof_resp=3)

print(f"Total Displacement at DOF 3: {result_pp*1000:.4f} mm (p-p)")

Total Displacement at DOF 3: 1.0412 mm (p-p)
